In [1]:

import os
from pathlib import Path
import sys
import numpy as np
import pickle
import pandas as pd
from lifelines import KaplanMeierFitter
import torch

pkg_path = str(Path(os.path.abspath('')).parent.absolute())
sys.path.insert(0, pkg_path)


data_path=pkg_path+'/results/'

from src import *

# use this if having problems:
# export PYTHONPATH=/home/ee577/project/src:$PYTHONPATH




In [2]:
pkg_path = str(Path(os.getcwd()).parent.absolute())  # Get parent directory of the current working directory
sys.path.insert(0, pkg_path)

# If you need to include a 'src' directory relative to the current notebook:
src_path = os.path.abspath(os.path.join(os.getcwd(), '../src'))  # Adjust path relative to the current working directory
sys.path.insert(0, src_path)

print(f"Path to the 'src' directory: {src_path}")

# Now you can import your modules from the 'src' directory
from src import *

# Load config file
config = global_config.config
config.device = 'cuda:1'
torch.manual_seed(config.seed)
print(f"This is the output directory: {config.dataset_dir}")

Path to the 'src' directory: /home/ee577/project/src
This is the output directory: /home/ee577/project/Datasets


In [3]:
preprocess_config = {
    'modality': ['DTI'], #['T2', 'FLAIR', 'T1', 'T1GD']
    'image_type': 'autosegm',
    'window': (140, 172, 164),
    'pad_window': (70, 86, 86),
    'crop': True,
    'window_idx': ((0, 144), (31, 214), (44,209)),
    'down_factor': 0.5,
    'augments': ['base'] #'base', 'flip', 'rotate', 'noise', 'deform'
}

# paths, image_dict,tumor_boxes=data_prep.convert_image_data_mod(**preprocess_config)


output_dir = '/home/ee577/project/Datasets/UPENN_GBM/DTI_numpy_files'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Iterate over each patient and their modalities in image_container
for pat, mod_arr in image_dict.items():
    # Create a directory for each patient
    patient_dir = os.path.join(output_dir, pat)
    if not os.path.exists(patient_dir):
        os.makedirs(patient_dir)

    # Check if 'AD' modality exists in the current patient's mod_arr
    if 'AD' in mod_arr:
        img_data = mod_arr['AD']  # Get the image data for the 'AD' modality
        
        # Construct the filename for the 'AD' modality (you can adjust the naming convention)
        filename = f"{pat}_AD.npy"
        filepath = os.path.join(patient_dir, filename)
        
        # Save the image data as a .npy file
        np.save(filepath, img_data)  # Save the image as a .npy file

        print(f"Saved image for patient {pat}, modality AD at {filepath}")
    else:
        print(f"No 'AD' modality found for patient {pat}. Skipping.")





In [4]:
# getting clinical data
clinical_info = pd.read_csv(os.path.join(config.upenn_dir, 'UPENN-GBM_clinical_info_v2.1.csv'))
#display(clinical_info)
print(clinical_info.columns)
print(list(clinical_info.dtypes))

pd.set_option("display.max_rows", 40)

survival_days = clinical_info['Survival_from_surgery_days_UPDATED']
survival_days = survival_days.drop(survival_days[survival_days == 'Not Available'].index)

print("statistics:")
display(survival_days.describe())

clinical_info.set_index('ID', inplace=True)

clinical_factors = clinical_info[['IDH1', 'MGMT']]

#list(features['Selected_features'].values)
features=pd.read_csv(data_path+'selected_features_DTI.csv')
all_features_DTI_AD_NC=pd.read_csv(os.path.join(config.upenn_dir,"csvs", 'Radiomic_Features_CaPTk_automaticsegm_DTI_AD_NC.csv'))

selected_features = list(features['Selected_features'].values)
matching_columns = [col for col in all_features_DTI_AD_NC.columns for feature in selected_features if feature in col]
filtered_data = all_features_DTI_AD_NC[matching_columns]
filtered_data_features = all_features_DTI_AD_NC[['SubjectID'] + matching_columns]
print(filtered_data_features.head())

filtered_data_features = filtered_data_features.set_index('SubjectID')
numerical_columns = filtered_data_features.select_dtypes(include=['number']).columns
if len(numerical_columns) == len(filtered_data_features.columns):
    print("All features are numerical.")
else:
    print("Some features are not numerical.")

Index(['ID', 'Gender', 'Age_at_scan_years',
       'Survival_from_surgery_days_UPDATED', 'Survival_Status',
       'Survival_Censor', 'IDH1', 'MGMT', 'KPS', 'GTR_over90percent',
       'Time_since_baseline_preop', 'PsP_TP_score'],
      dtype='object')
[dtype('O'), dtype('O'), dtype('float64'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('float64')]
statistics:


count     644
unique    471
top       376
freq        5
Name: Survival_from_surgery_days_UPDATED, dtype: object

            SubjectID  DTI_AD_NC_Intensity_Kurtosis  DTI_AD_NC_Intensity_Mean  \
0  UPENN-GBM-00001_11                      2.696694                 58.787582   
1  UPENN-GBM-00002_11                      1.605151                144.945654   
2  UPENN-GBM-00003_11                      6.648995                168.754045   
3  UPENN-GBM-00004_11                      3.575541                 90.657832   
4  UPENN-GBM-00005_11                      4.124732                 71.453383   

   DTI_AD_NC_Intensity_MeanAbsoluteDeviation  \
0                               8.498648e-15   
1                              -3.631480e-12   
2                              -7.904416e-14   
3                              -1.574397e-12   
4                               1.373632e-13   

   DTI_AD_NC_Intensity_MedianAbsoluteDeviation  DTI_AD_NC_Intensity_Minimum  \
0                                    -0.212418                           48   
1                                     6.945654                    

In [5]:
idh1_encoded = clinical_info['IDH1'].map({'Wildtype': 0, 'NOS/NEC': 1})
mgmt_encoded = clinical_info['MGMT'].map({'Methylated': 1, 'Unmethylated': 0})

idh1_encoded.fillna(idh1_encoded.mean(), inplace=True)
mgmt_encoded.fillna(mgmt_encoded.mean(), inplace=True)
encoded_df = pd.DataFrame({
    'IDH1_encoded': idh1_encoded,
    'MGMT_encoded': mgmt_encoded
})

merged_data = filtered_data_features.merge(encoded_df, left_index=True, right_index=True, how='left')

print(merged_data.head())

                    DTI_AD_NC_Intensity_Kurtosis  DTI_AD_NC_Intensity_Mean  \
SubjectID                                                                    
UPENN-GBM-00001_11                      2.696694                 58.787582   
UPENN-GBM-00002_11                      1.605151                144.945654   
UPENN-GBM-00003_11                      6.648995                168.754045   
UPENN-GBM-00004_11                      3.575541                 90.657832   
UPENN-GBM-00005_11                      4.124732                 71.453383   

                    DTI_AD_NC_Intensity_MeanAbsoluteDeviation  \
SubjectID                                                       
UPENN-GBM-00001_11                               8.498648e-15   
UPENN-GBM-00002_11                              -3.631480e-12   
UPENN-GBM-00003_11                              -7.904416e-14   
UPENN-GBM-00004_11                              -1.574397e-12   
UPENN-GBM-00005_11                               1.373632e-13  

In [26]:
# Step 1: Clean the survival days data
survival_days = clinical_info['Survival_from_surgery_days_UPDATED']
survival_days = pd.to_numeric(survival_days, errors='coerce')
survival_days = survival_days.dropna()  # Remove patients with missing data

# Step 3: Fit Kaplan-Meier Estimator
kmf = KaplanMeierFitter()
kmf.fit(durations=survival_days)

survival_probabilities = kmf.survival_function_

# Step 5: Define the survival probability intervals we are interested in
probabilities_to_bin = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
bins = []

# Loop through the probability ranges (lower bound to upper bound)
for i in range(len(probabilities_to_bin) - 1):
    lower_prob = probabilities_to_bin[i]
    upper_prob = probabilities_to_bin[i + 1]
    
    # Find the smallest time where the survival probability is just greater than or equal to the lower bound
    upper_time = survival_probabilities[survival_probabilities['KM_estimate'] <= lower_prob].index[0]
    
    # Find the smallest time where the survival probability is just greater than or equal to the upper bound
    lower_time = survival_probabilities[survival_probabilities['KM_estimate'] <= upper_prob].index[0]
    
    bins.append((lower_prob, upper_prob, upper_time, lower_time))  # Reverse time assignment (upper_time -> lower_prob)
    print(f"Survival probability range ({lower_prob}, {upper_prob}__Corresponds to {lower_time} {upper_time})")

# Step 6: Assign each patient to a survival bin based on their survival time
def assign_survival_bin(time, bins):
    """
    Assigns a survival bin to a patient based on their survival time.
    The bin is determined by the survival probability thresholds defined in 'bins'.
    """
    for lower_prob, upper_prob, upper_time, lower_time in bins:
        # Check if the patient's survival time falls within the bin's range
        if lower_time <= time < upper_time:
            return f"{lower_prob}-{upper_prob}"  # Return the range (e.g., "0.0-0.2")
    
    return "Above max bin"  # If the time doesn't fall in any bin (though it should)

binned_survival=clinical_info.copy()
# Step 7: Apply the binning function to the clinical data
binned_survival['survival_bin'] = survival_days.apply(lambda time: assign_survival_bin(time, bins))

# Check the result
print(binned_survival[['Survival_from_surgery_days_UPDATED', 'survival_bin']].head(20))


Survival probability range (0.0, 0.2__Corresponds to 709.0 6109.0)
Survival probability range (0.2, 0.4__Corresponds to 462.0 709.0)
Survival probability range (0.4, 0.6__Corresponds to 297.0 462.0)
Survival probability range (0.6, 0.8__Corresponds to 125.0 297.0)
Survival probability range (0.8, 1.0__Corresponds to 0.0 125.0)
                   Survival_from_surgery_days_UPDATED survival_bin
ID                                                                
UPENN-GBM-00001_11                                960      0.0-0.2
UPENN-GBM-00002_11                                291      0.6-0.8
UPENN-GBM-00003_11                               2838      0.0-0.2
UPENN-GBM-00004_11                                623      0.2-0.4
UPENN-GBM-00005_11                               1143      0.0-0.2
UPENN-GBM-00006_11                                626      0.2-0.4
UPENN-GBM-00007_11                                348      0.4-0.6
UPENN-GBM-00008_11                                469      0.2-0.4
U

In [7]:
def save_image_feature_pairing(image_containter, feature_df, output_file):
    """
    Function to load image data for a specific modality ('AD') and pair it with the corresponding feature data 
    for each patient, then save the paired data as a pickle file.

    Args:
        image_containter (OrderedDict): A dictionary of images for each patient and modality.
        feature_df (pd.DataFrame): A DataFrame containing the features for each patient.
        output_file (str): Path to the output pickle file.
    """
    # Initialize lists to hold image data and corresponding feature data
    X = []  # Image data
    y = []  # Feature data
    
    # Loop over each patient in the feature dataframe
    for patient_id, row in feature_df.iterrows():
        # Check if the patient has an 'AD' modality image available
        if patient_id in image_containter:
            # Retrieve the image data for 'AD' modality
            image_data = image_containter[patient_id].get('AD')
            if image_data is not None:
                # Add the image data (X) and the corresponding feature data (y)
                X.append(image_data)
                y.append(row.values)  # Assuming row contains the feature values
    
    # Convert the lists to numpy arrays
    X = np.array(X)
    y = np.array(y)

    # Save the image-feature pairs as a pickle file
    with open(output_file, 'wb') as f:
        pickle.dump((X, y), f)
    
    print(f"Saved paired data to {output_file}")




In [ ]:
out_dir = '/home/ee577/project/results/DTI_AD_NC_survival_data.pkl' 

save_image_feature_pairing(image_dict, binned_survival[['Survival_from_surgery_days_UPDATED', 'survival_bin']], out_dir)


NameError: name 'image_dict' is not defined